# Table of contents

0. [Setup](#setup)
1. [Segment Anything Model 3 (SAM 3)](#1)
2. [CLIP — Contrastive Language-Image](#2)
3. [Grounding DINO](#3)
4. [Homework](#4)
5. [References](#5)


# 0 - SETUP: Install SAM dependencies (run once)



## Option A: Using HuggingFace Transformers (recommended for this exercise)

In [ ]:

!pip install git+https://github.com/huggingface/transformers.git
!pip install accelerate torch torchvision matplotlib requests pillow
!pip install --upgrade transformers --break-system-packages

## Option B: Using Ultralytics (simpler API)


In [ ]:
!pip install -U ultralytics
!pip install git+https://github.com/ultralytics/CLIP.git

## Option C: Using official Meta repo


In [ ]:
!pip install sam3

## Authentication (required — request access at https://huggingface.co/facebook/sam3)


In [ ]:
!pip install huggingface_hub --upgrade

## HF authentication
Generate a HuggingFace token for read only. Add it as colab secret and export it as env variable.

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = str(userdata.get('HF_TOKEN'))

# 1 - Segment Anything Model 3 (SAM 3)



## 1.1 - Introduction

SAM 3 is the third generation of segmentation models released by Meta.  
It extends the capabilities introduced in SAM 2 and represents a major step toward generalized visual understanding and segmentation.

Built on top of the SAM 2 architecture, SAM 3 is the first unified segmentation framework capable of natively combining:

- Open-vocabulary understanding through natural language prompts
- Detection of multiple instances belonging to the same category
- High-precision pixel-level segmentation
- Multi-object video tracking and temporal consistency

| Model | Main Capability | Limitations |
|---|---|---|
| SAM | Promptable image segmentation | Limited semantic understanding |
| SAM 2 | Unified image and video segmentation | Limited concept-level reasoning |
| SAM 3 | Promptable Concept Segmentation (PCS) | Generalized multimodal segmentation |


<b><u>Unified Multimodal Segmentation</b></u>
---

SAM 3 combines several interaction modalities into a single architecture:

| Input Type | Supported |
|---|---|
| Points | Yes |
| Bounding Boxes | Yes |
| Masks | Yes |
| Text Prompts | Yes |
| Exemplar Images | Yes |
| Video Tracking | Yes |

This creates a highly flexible segmentation framework capable of adapting to many real-world scenarios.




## 1.2 Supported Prompt Types

SAM 3 supports multiple prompt modalities, including:

- Short text prompts (*noun phrases*)
- Cropped exemplar images
- A combination of both text and visual exemplars

Examples of supported prompts:

- `"red car"`
- `"construction worker"`
- `"damaged component"`
- An image crop containing the target object

This multimodal prompting system enables more robust and intuitive interaction.



---

### 1.2.1 Promptable Visual Segmentation (PVS)

Inherited from SAM 2, PVS allows segmentation through prompts such as:

- Points
- Bounding boxes
- Masks
- Scribbles

The user guides the segmentation process interactively.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_6_Clip_and_SAM/sam-ex1.gif" width=200>

### 1.2.2 Promptable Concept Segmentation (PCS)

SAM 3 introduces a new paradigm called: <b>*Promptable Concept Segmentation (PCS)*</b>

PCS expands segmentation beyond visual prompts by allowing the model to understand *concepts*.

The model can combine:

- Text prompts
- Exemplar images
- Visual prompts
- Semantic understanding

This allows segmentation based on abstract semantic meaning instead of only geometric guidance.

Examples:

- `"segment all recyclable materials"`
- `"find objects similar to this example"`
- `"detect unsafe regions"`
- `"segment all electronic components"`

SAM 3 operates over **semantic categories** instead of relying exclusively on fixed class labels.

The model understands high-level concepts and semantic relationships, enabling flexible segmentation across a wide variety of objects and scenarios.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_6_Clip_and_SAM/sam-ex3.gif" width=200>

---

### 1.2.3 Exemplar-Based Segmentation

SAM 3 introduces exemplar prompting.

Instead of defining a class explicitly, the user can provide one or more example objects.

The model then searches for semantically similar instances across the image or video.

Applications include:

- Industrial anomaly detection
- Medical pattern search
- Product recognition
- Few-shot segmentation

---

### 1.2.4 Open-Vocabulary Segmentation

SAM 3 is an **open-vocabulary** model.

This means it does **not** require:

* Retraining for new categories
* A predefined label set
* Closed-set classification constraints

SAM 3 can segment objects using natural language descriptions instead of relying only on predefined classes.

The model leverages semantic understanding learned during pretraining to generalize to previously unseen concepts.

Examples:

- `"segment all red cars"`
- `"detect every person wearing a helmet"`
- `"find damaged areas on the road"`

This enables zero-shot segmentation over previously unseen categories.

---

### 1.2.5 Zero-Shot Generalization

SAM 3 is capable of solving tasks in a: <b> *Zero-Shot Setting* </b>

The model can perform segmentation and tracking tasks without having seen explicit examples of those tasks during training.

This allows SAM 3 to generalize to:

* New object categories
* Novel environments
* Unseen semantic concepts
* Custom user-defined prompts

without additional fine-tuning.



### 1.2.6 Multi-Instance Retrieval with Consistent IDs

A single text prompt can retrieve **all matching instances** within an image or video.

Each detected object receives a unique identifier that remains temporally consistent across all video frames.

For example:

Input prompt:

```text
"players wearing blue jerseys"
````

SAM 3 can:

* Detect all matching players
* Segment each instance individually
* Assign persistent IDs
* Track them consistently throughout the video








---
## 1.3 Multi-Instance Segmentation

The model can identify and segment multiple objects belonging to the same semantic class simultaneously.

Examples:

- Multiple persons
- Multiple vehicles
- Multiple animals
- Multiple defects in industrial inspection

Unlike earlier approaches, SAM 3 maintains instance separation while preserving pixel-level accuracy.

---
## 1.4 Pixel-Level Precision

SAM 3 preserves the high-quality segmentation masks introduced by previous SAM models, allowing:

- Fine-grained object boundaries
- Accurate contour extraction
- Robust segmentation in complex scenes
- Detailed mask generation for downstream tasks

This is critical for:

- Medical imaging
- Autonomous driving
- Robotics
- Satellite imagery
- Industrial inspection

---

## 1.5 Video Object Tracking

SAM 3 extends SAM 2 video capabilities with stronger temporal understanding.

The model can:

- Track multiple objects across frames
- Maintain identity consistency
- Handle occlusions and motion changes
- Propagate masks efficiently through time

This enables advanced applications such as:

- Video surveillance
- Sports analytics
- Autonomous systems
- Human activity understanding

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_6_Clip_and_SAM/sam-ex2.gif" width=400>

## 1.6 Architecture

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_6_Clip_and_SAM/sam-ex4.gif" width=800>


### 1.6.1 Perception Encoder

The **Perception Encoder (PE)** is composed of:

- A **Vision Encoder**
- A **Text Encoder**

The **Perception Encoder** maps both images and text into the same high-dimensional vector space, allowing semantically similar concepts to remain close to each other regardless of whether they originate from visual or textual inputs.

This shared embedding space enables:

- Open-vocabulary understanding
- Cross-modal semantic reasoning
- Text-guided segmentation
- Exemplar matching
- Semantic similarity search

For example:

- The text `"dog"`
- An image of a dog
- A cropped exemplar of a dog

will be projected into nearby regions of the latent space.

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_6_Clip_and_SAM/sam3-pe.png" width=600/>


---

#### 1.6.1.1 Large-Scale Contrastive Pretraining

The Perception Encoder was pretrained using **contrastive learning** on approximately <b> *5.4 Billion Image-Text Pairs* </b>

During training, the model learns to:

- Bring semantically related image-text pairs closer together
- Push unrelated pairs farther apart in the embedding space

This enables strong open-vocabulary and zero-shot generalization capabilities.

---

#### 1.6.1.2 Difference Between PE and CLIP

Unlike CLIP, which generates a **single global embedding vector per image**, the Perception Encoder produces <b> Dense Region-Level Embeddings </b>.

Instead of representing the entire image with one vector, the PE generates embeddings for individual image regions.

This allows the model to understand not only:

- **What** is present in the image
- but also **Where** each concept is located

---

#### 1.6.1.3 Why Region-Level Embeddings Matter

This dense representation is critical for advanced segmentation tasks because it enables:

- Pixel-level localization
- Multi-instance detection
- Fine-grained semantic understanding
- Open-vocabulary segmentation
- Text-guided mask generation

For example, given the prompt:

```text
"red bicycle"

---

### 1.6.2 Shared Vision Encoder

The **Detector** and **Tracker** share the same vision backbone, known as the **Perception Encoder**.

This design allows both modules to operate over the same visual feature representations extracted from the image or video frames.

---

### 1.6.3 Decoupled Detector and Tracker

Although both components share the encoder, the **Detector** and **Tracker** are architecturally decoupled because they solve different problems.

| Component | Objective |
|---|---|
| Detector | Identify objects independently of identity |
| Tracker | Maintain and separate object identities across video frames |

The detector must remain **identity-agnostic**, while the tracker focuses on **identity consistency over time**.


---

### 1.6.4 Detector

#### 1.6.4.1 Fusion Encoder

The **Fusion Encoder** combines:

- Text prompts
- Image exemplars
- Visual prompts

into a unified set of <b> *Prompt Tokens* </b>.


These prompt tokens condition the model on the specific semantic concept the user wants to detect or segment.

---

#### 1.6.4.2 Concept-Aware Visual Features

The main role of the Fusion Encoder is to make the visual features become <b> *Concept-Aware* </b>.

Without the Fusion Encoder:

- The Vision Encoder produces generic image features
- The model describes everything visible in the scene equally
- No semantic priority is established

With the Fusion Encoder:

- The visual features are conditioned by the target concept
- The model focuses attention on semantically relevant regions
- The representation becomes task-specific and concept-specific

---

# Example

If the prompt is:

```text
"workers wearing helmets"
````

the Fusion Encoder modifies the visual representations so that regions related to:

* workers
* helmets
* protective equipment

receive stronger semantic emphasis.

This greatly improves:

* Open-vocabulary detection
* Semantic segmentation
* Multi-instance retrieval
* Tracking consistency


---

#### 1.6.4.3 DETR Decoder + Object Queries (ETR)

SAM 3 uses a DETR-style detection architecture based on:

* Transformer decoders
* Learnable object queries

This approach is commonly referred to as <b> ETR (DETR + Object Queries) </b>.

---

#### 1.6.4.4 Replacing Sliding Windows and Anchor Boxes

Traditional object detectors rely on:

* Sliding windows
* Anchor boxes
* Handcrafted proposal mechanisms

ETR replaces these approaches with <b> Object Queries </b>.

Object queries are learnable embedding vectors that compete with each other to discover objects in the scene.

Each query learns to specialize in detecting potential object instances directly from the transformer attention process.

---

# Advantages of Object Queries

| Traditional Detection   | DETR-Style Detection    |
| ----------------------- | ----------------------- |
| Anchor boxes            | Learnable queries       |
| Handcrafted heuristics  | End-to-end learning     |
| Complex post-processing | Simpler pipeline        |
| Dense proposals         | Sparse semantic queries |

This design enables:

* Cleaner architectures
* Better scalability
* Stronger semantic reasoning
* Improved multi-instance detection


---

#### 1.6.4.5 Presence Token

SAM 3 introduces a <b> Presence Token </b>.

A special global token added to the DETR decoder.

The Presence Token learns to predict whether the target concept is present in the image or video frame before detailed localization occurs.


<b> Why the Presence Token Matters </b>.

The Presence Token improves detection by separating:

| Task         | Purpose                         |
| ------------ | ------------------------------- |
| Recognition  | Determine if the concept exists |
| Localization | Determine where the concept is  |

This helps the model avoid unnecessary localization attempts when the target concept is absent.

<b> Benefits of the Presence Token </b>:

* Improves detection stability
* Reduces false positives
* Enhances semantic reasoning
* Improves open-vocabulary retrieval
* Strengthens video consistency

---

#### 1.6.4.6 Mask Head

The **Mask Head** converts detected object representations into:

# Pixel-Level Segmentation Masks

After the detector predicts object regions, the Mask Head generates precise segmentation masks for each instance.

This transforms:

* Bounding box detections
* Object queries
* Semantic features

into accurate per-pixel object masks.

<b> The Mask Head is responsible for: </b>

* Fine-grained contour extraction
* High-resolution mask prediction
* Instance separation
* Precise object boundaries

This enables applications requiring detailed segmentation accuracy such as:

* Medical imaging
* Robotics
* Autonomous driving
* Industrial inspection
* Video editing

---

### 1.6.5 Tracker Architecture

The tracker extends the:

<b>*Memory-Based Video Segmentation*</b>

framework inherited from SAM 2, while introducing additional temporal reasoning capabilities.

The tracker:

- Stores object representations in memory
- Propagates masks through time
- Maintains object identities
- Handles occlusions and motion changes
- Separates multiple tracked instances

This enables stable long-term video tracking.


#### 1.6.5.1 Advanced Temporal Tracking Pipeline

SAM 3 introduces several new tracking strategies designed to improve:

- Temporal consistency
- Identity preservation
- Long-term tracking stability
- Multi-object robustness

The tracker extends the memory-based video segmentation framework inherited from SAM 2 while integrating stronger temporal reasoning capabilities.


---

#### 1.6.5.2 Core Tracking Strategies

## Detect-Then-Propagate

SAM 3 adopts a:

# Detect → Then → Propagate

strategy.

The Detector is used to:

- Initialize new tracklets
- Confirm existing tracklets
- Recover lost objects
- Introduce new instances dynamically

After detection, the tracker propagates masks and identities through subsequent video frames.

This hybrid approach combines:

| Detector Strength | Tracker Strength |
|---|---|
| Spatial discovery | Temporal consistency |
| Semantic understanding | Identity preservation |
| Multi-instance retrieval | Motion continuity |


---

#### 1.6.5.3 Masklet Detection Score (MDS)

SAM 3 introduces the:

# Masklet Detection Score (MDS)

The MDS evaluates whether a propagated tracklet is still supported by the detector.

This mechanism helps eliminate:

- Spurious tracklets
- Drifted masks
- False-positive propagations
- Unstable temporal predictions

Only tracklets confirmed by the detector remain active over time.

---

#### 1.6.5.4 Periodic Re-Prompting

Long video sequences can accumulate tracking errors over time.

To mitigate this, SAM 3 performs:

# Periodic Re-Prompting

using high-confidence detections.

This process:

- Resets accumulated tracking drift
- Refreshes object representations
- Reinforces identity consistency
- Stabilizes long-term propagation

Essentially, the detector periodically "re-anchors" the tracker to reliable observations.


---

#### 1.6.5.5 Kalman Filter Integration

SAM 3 integrates a <b> Kalman Filter </b>

to improve motion prediction and identity preservation.

The Kalman Filter helps:

- Predict object trajectories
- Handle temporary occlusions
- Preserve identities during crossings
- Reduce identity swaps

This is especially important when multiple objects overlap or intersect on screen.

<b><u> Preventing Identity Swaps </u></b>

Consider two objects crossing paths:

- Two vehicles
- Two players
- Two animals

Without temporal motion modeling, the tracker may accidentally exchange identities.

The Kalman Filter provides motion continuity constraints that help maintain stable tracking identities across difficult scenes.


```text
DETECT → PROPAGATE → MATCH → UPDATE
   ↑                             ↑
Detector                  Kalman Filter
(find new objects)        (prevent identity swaps)
```





---

### 1.6.6 Detector vs Tracker

Although tightly connected, the Detector and Tracker solve fundamentally different problems.

<b><u> Detector: Spatial Understanding </b></u>

The Detector answers:

*"Where are all instances of the target concept in this frame?"*

Its responsibilities include:

- Discovering objects
- Segmenting instances
- Recognizing semantic concepts
- Handling open-vocabulary prompts

The detector operates independently on each frame.

<b><u> Tracker: Temporal Understanding </b></u>

The Tracker answers:

*"How do I know this object is the same one over time?"*

For example:

```text
"How do I know that the penguin in frame 100
is the same penguin from frame 1?"
```

The tracker focuses on:

- Identity preservation
- Temporal continuity
- Motion consistency
- Long-term object association

---

### 1.6.9 Why Share the Encoder?

Sharing the encoder is primarily an efficiency and engineering decision.

The image or video frame is encoded only once, and both the detector and tracker consume the same extracted features.

Without this shared architecture:

- Two large visual backbones would need to run independently
- GPU memory usage would increase significantly
- Inference latency would become much higher
- Computational redundancy would be introduced

By reusing the same feature representations, SAM 3 achieves:

- Lower computational cost
- Faster inference
- Better memory efficiency
- Cleaner multimodal integration

## 1.5 Hands On : Text-Prompted Concept Segmentation




### 1.5.1 Part 1 : PCS
---



- This exercise demonstrates Promptable Concept Segmentation (PCS)
- Given a text prompt, SAM 3 finds and segments ALL matching instances.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO
import os
from transformers import Sam3Processor, Sam3Model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = Sam3Processor.from_pretrained("facebook/sam3")
model = Sam3Model.from_pretrained("facebook/sam3").to(device)
model.eval()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")


#### Loading image as example
Using a COCO image with multiple objects of the same type


In [ ]:

image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

plt.figure(figsize=(10, 7))
plt.imshow(image)
plt.title("Input Image")
plt.axis("off")
plt.show()



#### Run text-prompted segmentation
- KEY CONCEPT: Unlike SAM 1/2 where you click on ONE object,
- SAM 3 takes a text prompt and finds ALL matching instances.

#### Visualization function for masks

In [ ]:
def visualize_masks(image, masks, scores, title="SAM 3 Results", show_original=False):
    """Overlay instance masks on the original image with distinct colors."""

      # Image with masks
    img_array = np.array(image).copy()
    colors = plt.cm.Set2(np.linspace(0, 1, max(len(masks), 1)))

    overlay = img_array.astype(float) / 255.0
    for i, (mask, score) in enumerate(zip(masks, scores)):
        mask_np = mask.cpu().numpy().astype(bool)
        color = colors[i % len(colors)][:3]
        for c in range(3):
            overlay[:, :, c] = np.where(
                mask_np,
                overlay[:, :, c] * 0.5 + color[c] * 0.5,
                overlay[:, :, c]
            )

    if show_original:
      fig, axes = plt.subplots(1, 2, figsize=(16, 7))

      axes[0].imshow(image)
      axes[0].set_title("Original Image")
      axes[0].axis("off")

      axes[1].imshow(overlay)
      axes[1].set_title(f"{title}\n({len(masks)} instances found)")
      axes[1].axis("off")
    else:
      plt.figure(figsize=(16, 7))
      plt.imshow(overlay)
      plt.title(f"{title}\n({len(masks)} instances found)")
      plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:


text_prompt = "cat"

# Question : What .to(device) line does ?
inputs = processor(images=image, text=text_prompt, return_tensors="pt").to(device)

# Question: Why are we disabling gradient in the next line ?
with torch.no_grad():
    outputs = model(**inputs) # Question : What is the porpouse of  **  in this line ?

# Post-process to get instance masks
results = processor.post_process_instance_segmentation(
    outputs,
    threshold=0.5,        # Confidence threshold for detection - Question : What means ?
    mask_threshold=0.5,   # Threshold for binary mask - Question : how it works ?
    target_sizes=inputs.get("original_sizes").tolist()
)[0]

print(f"\nText prompt: '{text_prompt}'")
print(f"Found {len(results['masks'])} instance(s)")

for i, (mask, score) in enumerate(zip(results["masks"], results["scores"])):
    print(f"  Instance {i+1}: confidence={score:.3f}, mask shape={mask.shape}")

visualize_masks(image, results["masks"], results["scores"],
                title=f'Text prompt: "{text_prompt}"')

### 1.5.2 Part 2 : Open-Vocabulary Exploration

- SAM 3 is not limited to COCO's 80 classes. Test creative prompts!
- The Presence Token will correctly return 0 instances for absent concepts.

In [ ]:

img_inputs = processor(images=image, return_tensors="pt").to(device)
with torch.no_grad():
    vision_embeds = model.get_vision_features(pixel_values=img_inputs.pixel_values)

test_prompts = [
    "cat",                    # Present (2 cats)
    "remote control",         # Present (remotes on table)
    "couch",                  # Present (1 couch)
    "cat ear",                # Fine-grained: parts of an object
    "yellow school bus",      # ABSENT — Presence Token should return 0
    "wooden table",           # Open vocabulary: not a COCO class
    "striped blanket",        # Creative description
]

print("=" * 60)
print("OPEN-VOCABULARY CONCEPT SEGMENTATION RESULTS")
print("=" * 60)

detected_masks_by_prompt = {}
for prompt in test_prompts:
    text_inputs = processor(text=prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(vision_embeds=vision_embeds, **text_inputs)

    results = processor.post_process_instance_segmentation(
        outputs, threshold=0.5, mask_threshold=0.5,
        target_sizes=img_inputs.get("original_sizes").tolist()
    )[0]

    n = len(results["masks"])
    status = "✓ FOUND" if n > 0 else "✗ ABSENT (Presence Token correctly filtered)"
    print(f'  "{prompt:25s}" → {n} instance(s)  {status}')

    detected_masks_by_prompt[prompt] = results

#### Display Images

In [ ]:
for p,r in detected_masks_by_prompt.items():
  visualize_masks(image, r["masks"], r["scores"],
                  title=f'Text prompt: "{p}"')

### 1.5.3 Part 3 : Video tracking


In [ ]:
!pip install gdown opencv-python

In [ ]:
# !wget --show-progress \
#     "https://huggingface.co/facebook/sam3/resolve/main/sam3.pt" \
#     -O "/content/sam3.pt"

from huggingface_hub import hf_hub_download

REPO_ID = "facebook/sam3"
FILENAME = "sam3.pt"

hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir="/content/")

In [ ]:

import gdown

file_id = "13d-WecRvyTozDcQrHxLV1t_e5GQzPiPz"
video_path = '/content/shinjuku.mp4'

url = f'https://drive.google.com/uc?id={file_id}'
gdown.download(url, video_path, quiet=False)


## Preprocessed Example mp4 file

ex_file_id = "1Yy74XqnVOCdRxBxSuw7o0X4iI0BL2Wav"
ex_video_path = '/content/shinjuku_preprocessed_ex.mp4'

url = f'https://drive.google.com/uc?id={ex_file_id}'
gdown.download(url, ex_video_path, quiet=False)

In [ ]:
from IPython.display import Video
Video("shinjuku.mp4", embed=True)

In [ ]:
from ultralytics.models.sam import SAM3VideoSemanticPredictor
from ultralytics import SAM

overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    model="sam3.pt",
    half=True,
    save=True,
)

predictor = SAM3VideoSemanticPredictor(overrides=overrides)

results = predictor(
    source=video_path,
    text=["person", "car"],
    stream=True
)

for i,r in enumerate(results):
    r.show()
    break

# results = predictor(
#     source="path/to/video.mp4",
#     bboxes=[[864, 383, 975, 620], [705, 229, 782, 402]],
#     labels=[1, 1],  # Positive labels
#     stream=True,
# )

#### Convert output to mp4

In [ ]:
import cv2
import os

output_path = "/content/shinjuku_masked.mp4"

cap    = cv2.VideoCapture(video_path)
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(f"Video: {width}×{height} @ {fps:.1f}fps | {total} frames")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print("Procesando...")

MAX_FRAMES = 200
total = MAX_FRAMES
for frame_idx, r in enumerate(results):
    if frame_idx >= MAX_FRAMES:
        break

    annotated = r.plot(
        masks=True,
        boxes=True,
        labels=True,
        conf=True,
        line_width=2,
    )

    writer.write(annotated)

    if frame_idx % 30 == 0:
        n = len(r.boxes) if r.boxes else 0
        pct = frame_idx / total * 100 if total > 0 else 0
        print(f" Frame {frame_idx}/{total} ({pct:.0f}%) — {n} objetos")

writer.release()
print(f"\n✅ Guardado en: {output_path}")

<img src="https://raw.githubusercontent.com/EzequielMatiasArevalo/tuia-computer-vision/refs/heads/main/media/pictures/class_6_Clip_and_SAM/sam3-mask-ex.gif" width=800>

# 2- CLIP — Contrastive Language-Image Pretraining



## 2.1 - Introduction

CLIP is a multimodal foundation model introduced by OpenAI that learns to connect images and natural language within a shared semantic embedding space.

Instead of training a classifier with a fixed set of labels, CLIP learns visual concepts directly from large-scale image-text pairs collected from the internet.

This enables:

- Open-vocabulary image understanding
- Zero-shot classification
- Image-text retrieval
- Semantic search
- Cross-modal reasoning


---

## 2.2 Core Idea

CLIP learns that:

- Images and their matching captions should have similar embeddings
- Unrelated image-text pairs should have distant embeddings

The model uses <b> Contrastive Learning </b> to align visual and textual representations.


---

## 2.3 High-Level Architecture

CLIP consists of two main encoders:

| Component | Purpose |
|---|---|
| Image Encoder | Extract visual features |
| Text Encoder | Extract language features |

Both encoders project their outputs into the same shared embedding space.

---

# CLIP Architecture

```text
                 ┌─────────────────┐
                 │     Image       │
                 └────────┬────────┘
                          ↓
                 ┌─────────────────┐
                 │  Image Encoder  │
                 │  (ViT / ResNet) │
                 └────────┬────────┘
                          ↓
                 Image Embedding
                          ↓
                   Shared Space
                          ↑
                  Text Embedding
                          ↑
                 ┌────────┴────────┐
                 │   Text Encoder  │
                 │   Transformer   │
                 └────────┬────────┘
                          ↓
                 ┌─────────────────┐
                 │      Text       │
                 └─────────────────┘
```


---

## 2.4 Image Encoder

The Image Encoder converts an image into a dense feature vector.

CLIP supports different backbone architectures:

| Backbone | Description |
|---|---|
| ResNet | CNN-based encoder |
| Vision Transformer (ViT) | Transformer-based encoder |

Modern CLIP variants commonly use Vision Transformers because of their superior representation quality.


---

## 2.5 Text Encoder

The Text Encoder is a Transformer-based language model.

It converts natural language prompts into semantic embeddings.

Example prompts:

```text
"a photo of a dog"
"a red sports car"
"a person riding a bicycle"
```

The resulting text embeddings are aligned with image embeddings in the shared latent space.

---

## 2.6 Shared Embedding Space

The key innovation in CLIP is the:

# Shared Multimodal Embedding Space

Images and text describing similar concepts are mapped close together.

Example:

| Input | Embedding Relationship |
|---|---|
| Image of a cat | Close to `"a cat"` |
| Image of a car | Close to `"a vehicle"` |
| Image of a dog | Far from `"airplane"` |

This enables semantic reasoning across modalities.


---

## 2.7 Contrastive Learning

CLIP is trained using a contrastive objective.

For a batch of image-text pairs:

- Matching pairs are pulled closer together
- Non-matching pairs are pushed farther apart


---

## 2.8 Zero-Shot Classification

One of CLIP’s most important capabilities is:

Zero-Shot Learning
---

The model can classify images into categories it never explicitly trained on.

Instead of retraining a classifier, we simply provide text prompts.

Example
---

Given an image:

```text
Image → ?
```

We compare it against prompts such as:

```text
"a photo of a dog"
"a photo of a cat"
"a photo of a truck"
```

The closest embedding determines the prediction.

No task-specific retraining is required.

---

## 2.9 Key Components

| Component | Function |
|---|---|
| Image Encoder | Extract visual features |
| Text Encoder | Extract language features |
| Shared Embedding Space | Align images and text |
| Contrastive Loss | Learn semantic similarity |
| Zero-Shot Inference | Generalize to unseen classes |


CLIP Training Pipeline
---

```text
Image + Caption Pair
          ↓
Image Encoder → Image Embedding
          ↓
Shared Semantic Space
          ↑
Text Embedding ← Text Encoder
          ↑
Caption
```

Contrastive learning aligns both embeddings.

---

## 2.10 Why CLIP Was Revolutionary ?

Traditional classifiers rely on:

- Fixed labels
- Supervised datasets
- Task-specific training

CLIP instead learns:

- Semantic concepts
- Natural language alignment
- Open-world generalization

This dramatically improves flexibility.


---

## 2.11 Common Use Cases


### 2.11.1 - Zero-Shot Image Classification

Recognize categories without retraining.

Example:

- Animals
- Vehicles
- Products
- Medical imagery

---

### 2.11.2 - Image Retrieval

Search images using text queries.

Example:

```text
"sunset over mountains"
```


---

### 2.11.3 Semantic Search

Search multimodal datasets using natural language.

Applications:

- Search engines
- Asset management
- Media indexing

---

### 2.11.4 Content Moderation

Detect unsafe or restricted content using semantic prompts.

---

### 2.11.5 - Visual Recommendation Systems

Match products or images semantically.

Example:

- Fashion recommendation
- Similar image retrieval
- Product discovery

---

## 2.12 CLIP vs Traditional CNN Classifiers

| Traditional CNN | CLIP |
|---|---|
| Fixed labels | Open vocabulary |
| Supervised only | Contrastive multimodal learning |
| Retraining required | Zero-shot capable |
| Image-only | Image + text |
| Limited generalization | Strong semantic generalization |


---

## 2.13 - CLIP Limitations

Although powerful, CLIP has several limitations:

| Limitation | Description |
|---|---|
| Global embeddings | Weak spatial localization |
| No segmentation masks | Cannot produce pixel-level outputs |
| Bias from web data | Learns internet-scale biases |
| Limited fine-grained reasoning | Sometimes struggles with subtle details |

These limitations motivated later systems such as:

- Grounded detection models
- SAM-style segmentation models
- Dense multimodal encoders


---

## 2.14 CLIP vs Dense Vision Models

| Model | Output Type |
|---|---|
| CLIP | One global image embedding |
| DINOv3 | Dense semantic embeddings |
| SAM 3 PE | Dense multimodal region embeddings |

CLIP focuses more on semantic alignment than precise localization.


---
## 2.15 Summary

CLIP is a multimodal vision-language model that:

- Aligns images and text
- Learns semantic concepts through contrastive learning
- Enables zero-shot image understanding
- Generalizes beyond fixed label sets

Its architecture consists of:

- An Image Encoder
- A Text Encoder
- A Shared Embedding Space
- A Contrastive Learning Objective

CLIP became one of the foundational architectures behind modern open-vocabulary and multimodal AI systems.


## 2.16 Hands On : CLIP

In [ ]:
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import torch
import requests
import matplotlib.pyplot as plt


image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

plt.figure(figsize=(10, 7))
plt.imshow(image)
plt.title("Input Image")
plt.axis("off")
plt.show()

In [ ]:

from transformers import pipeline


clip = pipeline(
   task="zero-shot-image-classification",
   model="openai/clip-vit-base-patch32",
   device=0
)
labels = [
    "a photo of a cat",
    "a photo of a dog",
    "a photo with a remote control"
]
clip(image, candidate_labels=labels)

# 3 - Grounding DINO


## 3.1 Introduction

:contentReference[oaicite:0]{index=0} is an open-vocabulary object detection model designed to detect arbitrary objects using natural language prompts.

Unlike traditional detectors trained on a fixed label set, Grounding DINO can localize objects described by free-form text such as:

```text
"traffic light"
"person wearing a helmet"
"red backpack"
"damaged component"
```

Grounding DINO combines:

- Vision Transformers
- Language understanding
- Cross-modal attention
- DETR-style detection

into a unified architecture for open-set object detection.

---

## 3.2 Main Goal

Grounding DINO solves the problem of:

# Open-Vocabulary Object Detection

It answers:

```text
"Where are the objects described by this text prompt?"
```

without requiring retraining for new categories.


---

## 3.3 Core Capabilities

| Capability | Description |
|---|---|
| Open-vocabulary detection | Detect unseen categories |
| Text-conditioned localization | Use natural language prompts |
| Zero-shot detection | No retraining required |
| Multi-instance detection | Detect multiple objects |
| Cross-modal reasoning | Align text and vision |
| End-to-end transformer detection | No handcrafted proposals |


---

## 3.4 High-Level Architecture

Grounding DINO combines:

- A Vision Encoder
- A Text Encoder
- A Feature Enhancer
- A Cross-Modal Fusion Module
- A DETR-style Decoder


---

## 3.5 Architecture Overview

```text
                ┌────────────────┐
                │     Image      │
                └───────┬────────┘
                        ↓
               ┌─────────────────┐
               │ Vision Encoder  │
               │     (ViT)       │
               └───────┬─────────┘
                       ↓
                Visual Features
                       ↓
             ┌──────────────────┐
             │ Feature Enhancer │
             └───────┬──────────┘
                     ↓
          Cross-Modal Fusion Module
                     ↑
             Text Features
                     ↑
             ┌───────────────┐
             │ Text Encoder  │
             │  (BERT/CLIP)  │
             └──────┬────────┘
                    ↑
               Text Prompt
                    ↓
          DETR Transformer Decoder
                    ↓
          Bounding Box Predictions
```

---

### 3.5.1 - Vision Encoder

The Vision Encoder extracts dense visual features from the input image.

Typical backbones include:

- Swin Transformer
- Vision Transformer (ViT)

Responsibilities:

- Encode spatial information
- Learn semantic representations
- Preserve object-level features

---

### 3.5.2- Text Encoder

The Text Encoder converts text prompts into semantic embeddings.

Example prompts:

```text
"a dog"
"blue helmet"
"construction worker"
```

The encoder enables semantic alignment between:

- Text concepts
- Visual regions

Common text encoders:

- BERT
- CLIP Text Encoder


---

### 3.5.3 - Feature Enhancer

The Feature Enhancer improves the quality of extracted visual features.

Responsibilities include:

- Multi-scale feature refinement
- Semantic enhancement
- Better object representation

This stage helps improve localization quality.

---

### 3.5.4 - Cross-Modal Fusion Module

This is one of the most important components.

The Cross-Modal Fusion Module combines:

- Visual features
- Text embeddings

to create <b> Text-Aware Visual Representations </b>.

This allows the detector to focus specifically on regions relevant to the text prompt.

---

### 3.5.5 -  DETR-Style Transformer Decoder

Grounding DINO uses a DETR-style decoder based on:

- Object queries
- Transformer attention
- End-to-end detection

Instead of:

- Sliding windows
- Anchor boxes
- Region proposal networks

the model uses <b> Learnable Object Queries </b>.

Each query competes to discover an object instance.


<u>Detection Pipeline</u>
---

```text
Object Queries
        ↓
Transformer Decoder
        ↓
Cross-Attention with Features
        ↓
Bounding Box Predictions
```

<u>Why DETR Matters</u>
---


DETR-style architectures simplify object detection by removing handcrafted components.

Advantages:

| Traditional Detection | DETR Detection |
|---|---|
| Anchor boxes | Learnable queries |
| NMS-heavy pipelines | End-to-end learning |
| Manual heuristics | Transformer reasoning |
| Dense proposals | Sparse semantic queries |

---

### 3.5.6 Grounding Mechanism

The term:

# "Grounding"

means associating language with specific image regions.

Example:

```text
"Find the bicycle"
```

The model grounds the text concept `"bicycle"` into:

- Bounding boxes
- Spatial locations
- Object instances

inside the image.

---

### 3.5.7 Open-Set Detection

Traditional object detectors can only detect categories seen during training.

Grounding DINO instead performs:

# Open-Set Detection

This means it can localize previously unseen categories using semantic understanding.



---

### 3.5.8 Zero-Shot Detection

Grounding DINO supports:

# Zero-Shot Generalization

The model can detect objects without task-specific retraining.

Example:

```text
"detect solar panels"
```

even if `"solar panels"` were not part of a predefined label set.

---

### 3.5.9 Training Strategy

Grounding DINO is trained using:

- Image-text datasets
- Detection datasets
- Contrastive alignment
- Transformer-based supervision

The model learns:

- Spatial localization
- Cross-modal alignment
- Semantic grounding

simultaneously.

### 3.5.10 Limitations

| Limitation | Description |
|---|---|
| Bounding boxes only | No native segmentation masks |
| High compute cost | Transformer-heavy architecture |
| Dense scenes remain challenging | Object overlap complexity |
| Temporal consistency not built-in | No native tracking |

## 3.6 Hands On : Example

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image
import torch

model_id = "IDEA-Research/grounding-dino-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

# Text prompt — separate categories with ". "
text_prompt = "cat. remote control. cat ears."

inputs = processor(images=image, text=text_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

# Post-process
results = processor.post_process_grounded_object_detection(
    outputs,
    inputs.input_ids,
    text_threshold=0.25,
    target_sizes=[image.size[::-1]]  # (H, W)
)[0]

for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    print(f"[{score:.3f}] {label:20s}  box={box.int().tolist()}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import numpy as np

def plot_detections(image: Image.Image, results: dict, figsize=(12, 8)):
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(image)

    colors = plt.cm.get_cmap("tab20").colors  # 20 distinct colors

    for i, (score, label, box) in enumerate(
        zip(results["scores"], results["labels"], results["boxes"])
    ):
        x1, y1, x2, y2 = box.int().tolist()
        w, h = x2 - x1, y2 - y1
        color = colors[i % len(colors)]

        # Draw box
        rect = mpatches.FancyBboxPatch(
            (x1, y1), w, h,
            boxstyle="square,pad=0",
            linewidth=2,
            edgecolor=color,
            facecolor="none"
        )
        ax.add_patch(rect)

        # Draw label background + text
        ax.text(
            x1, y1 - 6,
            f"{label}  {score:.2f}",
            fontsize=9, color="white", fontweight="bold",
            bbox=dict(facecolor=color, edgecolor="none", pad=2, alpha=0.85)
        )

    ax.axis("off")
    plt.tight_layout()
    plt.savefig("detections.png", dpi=150, bbox_inches="tight")
    plt.show()

# Usage
plot_detections(image, results)

# Exercises


Using the code presented by this notebook, answer the following questions :
- Q1: What happens when you prompt "animal" instead of "cat"? Does SAM 3 understand hypernyms (parent categories)?

- Q2: Try "sleeping cat" vs "sitting cat" — can SAM 3 distinguish the same object category with different attributes? Now try increasing the conffidence threshold to 0.95

- Q3: The Presence Token prevents phantom detections. Verify this: prompt "yellow school bus" and confirm 0 masks are returned. Now set threshold=0.01 — do phantom masks appear? What's happened with the instances of the rest of the classes ?

# Answer Section

### Part 1

- **Q: What does .to(device) do?**
- It moves all the tensors inside inputs to the target hardware — either "cuda" (GPU) or "cpu". PyTorch requires that the model and its inputs live on the same device, otherwise you get an error. Since you loaded the model onto device earlier, you must send the inputs there too. Think of it as making sure the data and the model are in the same "room" before running.

- **Q: Why are we disabling gradients?**
- Because you're doing inference, not training. During training, PyTorch builds a computation graph of every operation so it can later calculate gradients and update model weights. Here you're just asking the model for predictions — you'll never update any weights — so that graph is pure wasted memory and computation. torch.no_grad() turns it off, making inference faster and lighter on memory.

- Q: What is the purpose of ** in model(**inputs)?

*   Elemento de lista
*   Elemento de lista


- ** unpacks a dictionary into named keyword arguments. The processor returns inputs as a dictionary

- **Q: What does `threshold=0.5` mean?**

- This is the **detection confidence threshold**. The model assigns each detected segment a confidence score (0 to 1) representing how sure it is that the segment matches your text prompt `"cat"`. Any segment scoring below `0.5` gets thrown

**Q: How does `mask_threshold=0.5` work?**

The model doesn't output a clean black-and-white mask — it outputs a **soft probability map** where each pixel has a float value representing "how likely is this pixel part of the object." The `mask_threshold` binarizes it:
```
Raw pixel values:    0.2   0.7   0.9   0.4   0.6
After 0.5 threshold:  0     1     1     0     1
```
Pixels ≥ 0.5 become foreground (the cat), pixels below become background. Raising this threshold shrinks the mask edges; lowering it expands them.

### Part 2

- Q1: Yes, both masks are returned.
- Q2: Using pre-defined threshold of 0.5 both instance are still detected. Increasing the threshold to 0.95 only detects sleeping cats.
- Q3: Yellos school bus is not detected but returned handred of instances for the rest of the classes.


# References

- [Grounding DINO: Marrying DINO with Grounded
Pre-Training for Open-Set Object Detection](https://arxiv.org/pdf/2303.05499)
- [SAM 3: Segment Anything with Concepts](https://arxiv.org/pdf/2511.16719)
- [Learning Transferable Visual Models From Natural Language Supervision](https://arxiv.org/pdf/2103.00020)